In [4]:
import pandas as pd
from pathlib import Path
from scipy import stats

# where you saved the files
CORR_FILE   = Path(r"C:\Users\cdd\Documents\Uni\Special_course\df.csv")  
BEH_DIR    = Path(r"C:\Users\cdd\Documents\Uni\Special_course\ds003838-download")

SUBJECTS   = ["sub-034"]      # put all subjects here when you’re ready


In [5]:
corr = (pd.read_csv(CORR_FILE)
          .assign(                       # extract trial #, then +1 to match *.tsv
              trial=lambda d:
                  d.epoch.str.extract(r"trial_(\d+)\.csv")
                    .astype(int).squeeze()
                    + 1                 # 0-based → 1-based
          ))


In [6]:
def explode_triggers(df_beh):
    out_frames = []
    for _, row in df_beh.iterrows():
        trig = row["triggerCorrect"].strip()
        out_frames.append(pd.DataFrame({
            "trial"     : row["trial"],
            "digit_pos" : range(1, len(trig) + 1),   # 1, 2, …
            "recalled"  : [int(c) for c in trig]      # 0 / 1
        }))
    return pd.concat(out_frames, ignore_index=True)


In [8]:
subject_dfs = []      # collect for an optional grand-average
stats_per_sub = []    # store per-subject t-tests

for sub in corr["subject"].unique():
    # --- Behaviour ------------------------------------------------------
    beh_path = (BEH_DIR / sub / "beh" / f"{sub}_task-memory_beh.tsv")
    beh_raw  = pd.read_csv(beh_path, sep="\t", usecols=["trial", "triggerCorrect"], dtype = {"triggerCorrect": str})
    beh      = explode_triggers(beh_raw)

    # --- Merge ----------------------------------------------------------
    merged = corr.query("subject == @sub").merge(
                 beh, on=["trial", "digit_pos"], how="inner")

    # --- Stats ----------------------------------------------------------
    rec  = merged.loc[merged.recalled == 1, "r"]
    miss = merged.loc[merged.recalled == 0, "r"]

    t, p = stats.ttest_ind(rec, miss, equal_var=False)
    stats_per_sub.append(
        {"subject": sub,
         "mean_r_recalled": rec.mean(),
         "mean_r_missed"  : miss.mean(),
         "t" : t, "p" : p, "n_recalled": len(rec), "n_missed": len(miss)}
    )

    merged["subject"] = sub
    subject_dfs.append(merged)


In [9]:
pd.set_option("display.precision", 3)
print(pd.DataFrame(stats_per_sub))

    subject  mean_r_recalled  mean_r_missed      t          p  n_recalled  \
0   sub-033            0.322          0.199  6.330  4.212e-10         357   
1   sub-034            0.240          0.214    NaN        NaN         567   
2   sub-035            0.425          0.372  0.775  4.424e-01          75   
3   sub-036            0.398          0.292  3.097  2.314e-03          89   
4   sub-038            0.295          0.225  2.208  2.811e-02         159   
5   sub-039            0.317          0.216  4.219  2.818e-05         360   
6   sub-040            0.302          0.202  4.034  6.259e-05         385   
7   sub-041            0.304          0.226  4.172  3.298e-05         448   
8   sub-042            0.252          0.203    NaN        NaN         532   
9   sub-043            0.278          0.216  2.766  5.811e-03         363   
10  sub-044            0.296          0.257  2.107  3.544e-02         426   
11  sub-045            0.312          0.259  2.089  3.734e-02         203   

In [11]:
print(f"sub {sub}:  n_recalled = {len(rec)},  var = {rec.var(ddof=1):.3g}")
print(f"sub {sub}:  n_missed   = {len(miss)}, var = {miss.var(ddof=1):.3g}")


sub sub-098:  n_recalled = 173,  var = 0.0699
sub sub-098:  n_missed   = 214, var = 0.0701


In [12]:
import pandas as pd
from scipy import stats

# concatenate everything the loop stored
group_df = pd.concat(subject_dfs, ignore_index=True)

# two samples
rec_all  = group_df.loc[group_df.recalled == 1, "r"]
miss_all = group_df.loc[group_df.recalled == 0, "r"]

# Welch two-sample t-test → one-tailed (recalled > missed)
t_stat, p_two = stats.ttest_ind(rec_all, miss_all, equal_var=False, nan_policy="omit")
p_one = p_two / 2 if t_stat > 0 else 1 - p_two / 2

print(f"Pooled: t = {t_stat:.3f}, one-tailed p = {p_one:.4g}, "
      f"mean Δr = {rec_all.mean() - miss_all.mean():.3f}  (N = {len(rec_all)+len(miss_all)})")


Pooled: t = 20.129, one-tailed p = 6.008e-90, mean Δr = 0.063  (N = 39642)


In [13]:
# 1) build a table of means per subject × recalled(0/1)
sub_means = (group_df
             .groupby(["subject", "recalled"])["r"]
             .mean()
             .unstack())        # columns 0 = missed, 1 = recalled

# 2) drop any subject lacking one category (all-correct or all-wrong)
sub_means = sub_means.dropna(subset=[0, 1])

# 3) paired t-test (recalled − missed)
from scipy.stats import ttest_rel
t_stat, p_two = ttest_rel(sub_means[1], sub_means[0])
p_one = p_two / 2 if t_stat > 0 else 1 - p_two / 2

print(f"Paired: t = {t_stat:.3f}, one-tailed p = {p_one:.4g}, "
      f"mean Δr = {(sub_means[1]-sub_means[0]).mean():.3f}  (N_subjects = {len(sub_means)})")


Paired: t = 13.328, one-tailed p = 3.646e-19, mean Δr = 0.072  (N_subjects = 56)
